# Panchayat Data Cleaning Pipeline — Phase 1 (Review 2)

**How to use:**
1. Run the first cell to mount Google Drive.
2. Set `BASE_DIR` to the path of your uploaded `Data_Organized` folder.
3. Run the remaining cells top to bottom.
4. Cleaned outputs are written to `<BASE_DIR>/Cleaned/<dataset_name>.csv`.

**Notebook structure**

| Part | Dataset | Notes |
|---|---|---|
| 0 | Setup & default cleaner | Reused by most datasets below |
| 1 | Census | Plain `default_clean` |
| 2 | Education | Strips Excel "force text" apostrophe from code columns |
| 3 | Finance | 4 files are HTML tables disguised as `.xls` — custom loader |
| 4 | Health | `hospital_directory.csv` geography mismatch flagged, filtered to Karnataka |
| 5 | Infrastructure & Sanitation | Plain `default_clean` |
| 6 | GP_PDP_Reports | Custom parser: ~30 per-GP score sheets → one tidy long table |
| 7 | GPDP_Planning | Multi-row stacked headers handled manually |
| 8 | Reference_Master_Data | Plain `default_clean` |
| 9 | Run everything | Executes all of the above in order |


In [ ]:
# Mount Google Drive so BASE_DIR can point at your uploaded Data_Organized folder
from google.colab import drive
drive.mount('/content/drive')


Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


## Part 0 — Setup & generic (default) cleaning helpers

In [ ]:
import pandas as pd
import numpy as np
import re
import os
import warnings
warnings.filterwarnings("ignore")

BASE_DIR = "/content/drive/MyDrive/Colab Notebooks/Data_Organized"
OUT_DIR = os.path.join(BASE_DIR, "Cleaned")
os.makedirs(OUT_DIR, exist_ok=True)

# Tokens that should all be treated as missing, regardless of dataset
NA_TOKENS = ["", "NA", "N/A", "na", "n/a", "NIL", "Nil", "nil", "-", "--",
             "None", "none", "NULL", "null", "N.A.", "n.a."]

def save(df, name):
    """Save a cleaned dataframe and print a one-line confirmation."""
    path = os.path.join(OUT_DIR, f"{name}.csv")
    df.to_csv(path, index=False)
    print(f"  -> saved {name}: {df.shape[0]} rows x {df.shape[1]} cols -> {path}")

def clean_column_names(df):
    """Strip whitespace / newlines from headers. Keeps original wording."""
    df = df.copy()
    df.columns = [re.sub(r"\s+", " ", str(c)).strip() for c in df.columns]
    return df

def strip_string_cells(df):
    """Trim whitespace on every text column (this is the pandas .str.strip()
    step from the tutorials, applied dataset-wide instead of column-by-column)."""
    df = df.copy()
    obj_cols = df.select_dtypes(include="object").columns
    for c in obj_cols:
        df[c] = df[c].astype(str).str.strip()
        df[c] = df[c].replace({tok: np.nan for tok in NA_TOKENS})
        df.loc[df[c].isin(["nan", "NaN"]), c] = np.nan
    return df

def default_clean(df, name="", garbage_pattern=r"^\?+$"):
    """
    General-purpose cleaner for any 'one header row + one row per record' file.
    Steps (matches the standard tutorial workflow):
      1. standardize column names
      2. strip whitespace / standardize NA tokens on text columns
      3. replace garbage placeholder values (e.g. '????????????') with NaN
      4. drop exact duplicate rows
      5. drop columns that are 100% empty
      6. reset index
    Returns the cleaned df plus a small report dict.
    """
    before_rows, before_cols = df.shape
    df = clean_column_names(df)
    df = strip_string_cells(df)

    obj_cols = df.select_dtypes(include="object").columns
    for c in obj_cols:
        df[c] = df[c].replace(regex={garbage_pattern: np.nan})

    dup_count = df.duplicated().sum()
    df = df.drop_duplicates()

    empty_cols = [c for c in df.columns if df[c].isna().all()]
    df = df.drop(columns=empty_cols)

    df = df.reset_index(drop=True)

    report = {
        "name": name,
        "rows_before": before_rows, "rows_after": df.shape[0],
        "cols_before": before_cols, "cols_after": df.shape[1],
        "duplicates_dropped": int(dup_count),
        "empty_columns_dropped": empty_cols,
    }
    print(f"[{name}] {before_rows}x{before_cols} -> {df.shape[0]}x{df.shape[1]} "
          f"(dropped {dup_count} dup rows, {len(empty_cols)} empty cols)")
    return df, report

def missing_value_report(df, name):
    """Quick % missing per column — use this before deciding impute vs drop."""
    pct = (df.isna().sum() / len(df) * 100).round(1)
    pct = pct[pct > 0].sort_values(ascending=False)
    if len(pct):
        print(f"[{name}] columns with missing values (%):")
        print(pct.to_string())
    else:
        print(f"[{name}] no missing values.")
    return pct



## Part 1 — Census

In [ ]:
def clean_census():
    print("\n=== CENSUS ===")
    results = {}

    df = pd.read_excel(os.path.join(BASE_DIR, "Census", "DDW_PCA2905_2011_MDDS with UI.xlsx"))
    df, rep = default_clean(df, "census_ddw_pca2905")
    save(df, "census_ddw_pca2905"); results["ddw"] = df

    df = pd.read_excel(os.path.join(BASE_DIR, "Census", "PCA_CDB_2929_F_Census (2).xls"))
    df, rep = default_clean(df, "census_pca_cdb_2929")
    save(df, "census_pca_cdb_2929"); results["pca_cdb"] = df

    df = pd.read_csv(os.path.join(BASE_DIR, "Census", "raichur_village_to_GP_mapping.csv"))
    df, rep = default_clean(df, "raichur_village_to_gp_mapping")
    # gp_lgd_code / village_code are identifiers, not quantities -> keep as
    # nullable integer, not float, so they don't get "219671.0"-style codes
    for c in ["village_code", "gp_lgd_code"]:
        if c in df.columns:
            df[c] = pd.to_numeric(df[c], errors="coerce").astype("Int64")
    save(df, "raichur_village_to_gp_mapping"); results["village_to_gp"] = df

    df = pd.read_csv(os.path.join(BASE_DIR, "Census", "Raichur_GP_Level_Census_2011",
                                   "raichur_GP_level_census2011.csv"))
    df, rep = default_clean(df, "raichur_gp_level_census2011")
    if "gp_lgd_code" in df.columns:
        df["gp_lgd_code"] = pd.to_numeric(df["gp_lgd_code"], errors="coerce").astype("Int64")
    save(df, "raichur_gp_level_census2011"); results["gp_census"] = df

    df = pd.read_csv(os.path.join(BASE_DIR, "Census", "Raichur_GP_Level_Census_2011",
                                   "raichur_villages_unmatched_to_census2011.csv"))
    df, rep = default_clean(df, "raichur_villages_unmatched")
    save(df, "raichur_villages_unmatched"); results["unmatched"] = df

    return results



## Part 2 — Education

In [ ]:
def clean_education():
    """
    EDUCATION.csv is wide (396 cols) and India-govt-portal identifier columns
    (State Code, District Code, Sub District Code, Village Code, CD Block Code)
    are stored as text with a leading apostrophe, e.g.  '29   '572   '05542
    That apostrophe is an Excel "force text" marker, not part of the code —
    plain default_clean would leave it in and break any join against
    Census/Master_Reference_Data codes that don't have it.
    """
    print("\n=== EDUCATION ===")
    df = pd.read_csv(os.path.join(BASE_DIR, "Education", "EDUCATION.csv"), low_memory=False)
    df, rep = default_clean(df, "education")

    code_cols = [c for c in df.columns if c.strip().endswith("Code")]
    for c in code_cols:
        df[c] = df[c].astype(str).str.replace("'", "", regex=False).str.strip()
        df.loc[df[c].isin(["nan", ""]), c] = np.nan

    missing_value_report(df, "education")
    save(df, "education")
    return df



## Part 3 — Finance  (mixed: plain CSV/XLSX + HTML-disguised-as-.xls files)

In [ ]:
def _read_govt_html_xls(path):
    """
    Loader for the eGramSwaraj-style .xls exports that are actually saved
    HTML tables with 2-3 stacked header rows. pd.read_excel() fails on these
    with 'Excel file format cannot be determined' — use pd.read_html instead,
    then flatten whatever multi-row header pandas inferred.
    """
    tables = pd.read_html(path)
    df = tables[0]
    if isinstance(df.columns, pd.MultiIndex):
        flat_cols = []
        for tup in df.columns:
            parts = [str(p).strip() for p in tup
                     if str(p).strip() and not str(p).strip().startswith("Unnamed")]
            flat_cols.append(" - ".join(dict.fromkeys(parts)) if parts else "col")
        df.columns = flat_cols
    return df

def clean_finance():
    print("\n=== FINANCE ===")
    results = {}
    finance_dir = os.path.join(BASE_DIR, "Finance")

    # --- HTML-disguised .xls reports: custom loader, then default_clean ---
    html_xls_files = {
        "activity_status_report": "Activity Status Report.xls",
        "activity_wise_expenditure_report": "Activity wise Expenditure Report.xls",
        "approved_action_plan_report": "Approved Action Plan Report.xls",
        "egramswaraj_investment_expenditure": "eGramSwaraj investment:expenditure.xls",
    }
    for out_name, fname in html_xls_files.items():
        df = _read_govt_html_xls(os.path.join(finance_dir, fname))
        df, rep = default_clean(df, out_name)
        save(df, out_name)
        results[out_name] = df

    # --- ReceiptHeadReport.xlsx: normal xlsx, but Particulars has literal
    #     "????????????" placeholders where the source ID was unreadable ---
    df = pd.read_excel(os.path.join(finance_dir, "ReceiptHeadReport.xlsx"))
    df, rep = default_clean(df, "receipt_head_report", garbage_pattern=r"^\?{4,}$")
    save(df, "receipt_head_report")
    results["receipt_head_report"] = df

    # --- Comprehensive Activity Details Report_All.csv: plain default clean.
    #     NOTE: your teammates are still finishing this + Activity wise
    #     Expenditure — re-run this cell once they hand off the final file,
    #     the code doesn't need to change. ---
    df = pd.read_csv(os.path.join(finance_dir, "Comprehensive Activity Details Report_All.csv"),
                      low_memory=False)
    df, rep = default_clean(df, "comprehensive_activity_details")
    save(df, "comprehensive_activity_details")
    results["comprehensive_activity"] = df

    # --- Historical_Data panels: plain default clean, keep gp_lgd_code as int ---
    hist_dir = os.path.join(finance_dir, "Historical_Data")
    for fname, out_name in [
        ("devadurga_XVFC_finance_panel_2022_2026.csv", "devadurga_xvfc_finance_panel"),
        ("devadurga_overall_receipts_payments_panel_2022_2026.csv", "devadurga_receipts_payments_panel"),
    ]:
        df = pd.read_csv(os.path.join(hist_dir, fname))
        df, rep = default_clean(df, out_name)
        if "gp_lgd_code" in df.columns:
            df["gp_lgd_code"] = pd.to_numeric(df["gp_lgd_code"], errors="coerce").astype("Int64")
        save(df, out_name)
        results[out_name] = df

    return results



## Part 4 — Health  (geography mismatch is flagged, not papered over)

In [ ]:
def clean_health():
    print("\n=== HEALTH ===")
    results = {}
    health_dir = os.path.join(BASE_DIR, "Health")

    df = pd.read_csv(os.path.join(health_dir, "hospital_directory.csv"), low_memory=False)
    df, rep = default_clean(df, "hospital_directory_raw")

    n_total = len(df)
    n_karnataka = (df["State"].astype(str).str.strip().str.lower() == "karnataka").sum()
    print(f"[hospital_directory] {n_karnataka}/{n_total} rows are State == 'Karnataka'.")
    print("  This file is a national/multi-state hospital directory, not a "
          "Karnataka-only dataset. Filtering to Karnataka below — if that "
          "leaves too few usable rows for your health indicator, flag it to "
          "your guide as a documented data gap rather than silently using "
          "the full (mostly non-Karnataka) file.")
    df_ka = df[df["State"].astype(str).str.strip().str.lower() == "karnataka"].reset_index(drop=True)
    save(df, "hospital_directory_all_states")       # full cleaned file, for reference
    save(df_ka, "hospital_directory_karnataka_only")  # the one to actually use
    results["hospital_directory_all"] = df
    results["hospital_directory_karnataka"] = df_ka

    df = pd.read_csv(os.path.join(health_dir, "HealthInfrastructure_Davanager.csv"))
    df, rep = default_clean(df, "health_infrastructure_davanagere")
    save(df, "health_infrastructure_davanagere")
    results["health_infra_davanagere"] = df

    return results



## Part 5 — Infrastructure & Sanitation

In [ ]:
def clean_infrastructure_sanitation():
    print("\n=== INFRASTRUCTURE & SANITATION ===")
    results = {}
    infra_dir = os.path.join(BASE_DIR, "Infrastructure_Sanitation")

    df = pd.read_csv(os.path.join(infra_dir, "SANITATION AND INFRASTRUCTURE.csv"), low_memory=False)
    df, rep = default_clean(df, "sanitation_and_infrastructure")
    if "gp_code" in df.columns:
        df["gp_code"] = pd.to_numeric(df["gp_code"], errors="coerce").astype("Int64")
    save(df, "sanitation_and_infrastructure")
    results["sanitation"] = df

    # NOTE: "Village Level Drinking water.csv" was removed because it was a
    # byte-for-byte duplicate of SANITATION AND INFRASTRUCTURE.csv — that
    # sanitation file is now the single source for both indicators. If your
    # water-supply columns (tap_*, hand pump, etc.) live inside it, pull them
    # from `results["sanitation"]` rather than re-sourcing a second file.

    df = pd.read_excel(os.path.join(infra_dir, "Gabbur_Road_Condition.xlsx"))
    df, rep = default_clean(df, "gabbur_road_condition")
    save(df, "gabbur_road_condition")
    results["road_condition"] = df

    return results



## Part 6 — GP_PDP_Reports  (custom parser: per-GP score sheets -> one tidy table)

In [ ]:
def _parse_gp_pdp_sheet(path):
    """
    Each per-GP file has this fixed shape:
      row0: header row  -> Financial Year | Grade | State Name | Zilla ... | Block ... | GP name
      row1: values for the header above (this IS the metadata we want)
      row2: blank
      row3: header row for the indicator table -> S.No. | Indicator | Total Score | Score Obtained
      row4..N-2: one row per indicator
      last row: 'Total' summary row
    Returns (metadata_dict, indicators_df).
    """
    raw = pd.read_excel(path, header=None)
    meta_keys = raw.iloc[0, :6].tolist()
    meta_vals = raw.iloc[1, :6].tolist()
    meta = dict(zip(meta_keys, meta_vals))

    table = raw.iloc[3:, :4].copy()
    table.columns = table.iloc[0]
    table = table.iloc[1:].reset_index(drop=True)
    table = table.rename(columns=lambda c: str(c).strip())

    total_row = table[table["S.No."].astype(str).str.strip().str.lower() == "total"]
    indicators = table[table["S.No."].astype(str).str.strip().str.lower() != "total"].copy()

    indicators["Total Score"] = pd.to_numeric(indicators["Total Score"], errors="coerce")
    indicators["Score Obtained"] = pd.to_numeric(indicators["Score Obtained"], errors="coerce")
    indicators["gp_name"] = meta.get("Gram Panchayat and Equivalent")

    total_score_sum = float(total_row["Score Obtained"].iloc[0]) if len(total_row) else np.nan
    meta["reported_total_score"] = total_score_sum
    return meta, indicators

def clean_gp_pdp_reports():
    """
    Combines all per-GP PDP score sheets into two tidy tables:
      - gp_pdp_scores_long:  one row per (GP, indicator) — good for analysis
      - gp_pdp_scores_summary: one row per GP with grade + total score,
        cross-checked against Devadurga_GPDP_Master.csv
    The "PDP_Assessment_Report (1/2/3).xlsx" files are grade-band rosters
    (Beginner/Aspirant/Performer) — a different layout (Zilla-level roll-up,
    not per-GP), so they're parsed separately and NOT mixed into the per-GP
    long table.
    NOTE ON FILE LOCATIONS (matches the cleaned-up folder layout):
      - The generic-named duplicate of Gabbur.xlsx ("PDP_Assessment_Report.xlsx",
        no number suffix) has already been removed from GP_PDP_Reports/.
      - The three grade-band roster files now live at the TOP LEVEL of
        BASE_DIR (Data_Organized/PDP_Assessment_Report (1/2/3).xlsx),
        not inside GP_PDP_Reports/ — the paths below reflect that.
    """
    print("\n=== GP_PDP_REPORTS ===")
    pdp_dir = os.path.join(BASE_DIR, "GP_PDP_Reports")

    per_gp_files = [f for f in os.listdir(pdp_dir) if f.endswith(".xlsx")]

    all_meta, all_indicators = [], []
    for fname in sorted(per_gp_files):
        meta, indicators = _parse_gp_pdp_sheet(os.path.join(pdp_dir, fname))
        all_meta.append(meta)
        all_indicators.append(indicators)

    long_df = pd.concat(all_indicators, ignore_index=True)
    long_df, _ = default_clean(long_df, "gp_pdp_scores_long")
    save(long_df, "gp_pdp_scores_long")

    summary_df = pd.DataFrame(all_meta)
    summary_df, _ = default_clean(summary_df, "gp_pdp_scores_summary")
    save(summary_df, "gp_pdp_scores_summary")

    # Cross-check against the master file
    master = pd.read_csv(os.path.join(pdp_dir, "Devadurga_GPDP_Master.csv"))
    master, _ = default_clean(master, "devadurga_gpdp_master")
    save(master, "devadurga_gpdp_master")

    merged = summary_df.merge(
        master, left_on="Gram Panchayat and Equivalent", right_on="GP Name", how="outer",
        indicator=True)
    mismatches = merged[merged["_merge"] != "both"]
    if len(mismatches):
        print(f"[gp_pdp check] {len(mismatches)} GP name(s) did not match between the "
              f"per-GP files and Devadurga_GPDP_Master.csv — check spelling variants:")
        print(mismatches[["Gram Panchayat and Equivalent", "GP Name", "_merge"]].to_string())

    # Grade-band roster files (different layout: metadata rows 0-1, blank row 2,
    # real header on row 3, data from row 4 -- same offset as the per-GP files).
    # These live at the top level of BASE_DIR, not inside GP_PDP_Reports/.
    roster_files = sorted(f for f in os.listdir(BASE_DIR)
                           if re.match(r"^PDP_Assessment_Report \(\d+\)\.xlsx$", f))
    roster_frames = []
    for fname in roster_files:
        raw = pd.read_excel(os.path.join(BASE_DIR, fname), header=None)
        grade_label = raw.iloc[1, 1]          # e.g. "Beginner (0-40) D"
        body = raw.iloc[3:].copy()
        body.columns = body.iloc[0]
        body = body.iloc[1:].reset_index(drop=True)
        body["grade_band"] = grade_label
        roster_frames.append(body)
    roster_df = pd.concat(roster_frames, ignore_index=True)
    roster_df, _ = default_clean(roster_df, "pdp_grade_band_roster")
    save(roster_df, "pdp_grade_band_roster")

    return {"long": long_df, "summary": summary_df, "master": master, "roster": roster_df}



## Part 7 — GPDP_Planning  (multi-row headers)

In [ ]:
def clean_gpdp_planning():
    print("\n=== GPDP_PLANNING ===")
    results = {}
    planning_dir = os.path.join(BASE_DIR, "GPDP_Planning")

    # Scheme_Wise_Report.xlsx: rows 0-3 are metadata (year/state/level),
    # rows 4-5 are a 2-row stacked header, data starts row 7 (row 6 is blank)
    raw = pd.read_excel(os.path.join(planning_dir, "Scheme_Wise_Report.xlsx"), header=None)
    financial_year = raw.iloc[0, 0]
    state = raw.iloc[1, 0]
    level = raw.iloc[2, 0]

    top = raw.iloc[4].ffill()          # forward-fill merged top header cells
    bottom = raw.iloc[5]
    cols = []
    for t, b in zip(top, bottom):
        t, b = str(t).strip(), str(b).strip()
        if b and b.lower() != "nan":
            cols.append(f"{t} - {b}" if t.lower() != "nan" else b)
        else:
            cols.append(t)
    df = raw.iloc[7:].copy()
    df.columns = cols
    df["financial_year_meta"] = financial_year
    df["state_meta"] = state
    df["panchayat_level_meta"] = level
    df, rep = default_clean(df, "scheme_wise_report")
    save(df, "scheme_wise_report")
    results["scheme_wise_report"] = df

    # Gabbur_Project_Feasibility.xlsx: simple 3-column key/value table
    df = pd.read_excel(os.path.join(planning_dir, "Gabbur_Project_Feasibility.xlsx"))
    df, rep = default_clean(df, "gabbur_project_feasibility")
    save(df, "gabbur_project_feasibility")
    results["project_feasibility"] = df

    return results



## Part 8 — Reference_Master_Data

In [ ]:
def clean_reference_master_data():
    print("\n=== REFERENCE_MASTER_DATA ===")
    results = {}
    ref_dir = os.path.join(BASE_DIR, "Reference_Master_Data", "Master_Reference_Data")

    df = pd.read_csv(os.path.join(ref_dir, "karnataka_GP_master_statewide.csv"))
    df, rep = default_clean(df, "karnataka_gp_master_statewide")
    if "gp_lgd_code" in df.columns:
        df["gp_lgd_code"] = pd.to_numeric(df["gp_lgd_code"], errors="coerce").astype("Int64")
    save(df, "karnataka_gp_master_statewide")
    results["gp_master"] = df

    df = pd.read_csv(os.path.join(ref_dir, "karnataka_district_code_reference.csv"))
    df, rep = default_clean(df, "karnataka_district_code_reference")
    save(df, "karnataka_district_code_reference")
    results["district_code_ref"] = df

    df = pd.read_csv(os.path.join(ref_dir, "raichur_excluded_ULB_and_incomplete_rows.csv"))
    df, rep = default_clean(df, "raichur_excluded_ulb_rows")
    save(df, "raichur_excluded_ulb_rows")
    results["excluded_ulb"] = df

    return results



## Part 9 — Run everything

In [ ]:
if __name__ == "__main__":
    census_results = clean_census()
    education_result = clean_education()
    finance_results = clean_finance()
    health_results = clean_health()
    infra_results = clean_infrastructure_sanitation()
    gp_pdp_results = clean_gp_pdp_reports()
    planning_results = clean_gpdp_planning()
    reference_results = clean_reference_master_data()

    print("\n=== ALL DATASETS CLEANED. Files written to:", OUT_DIR, "===")



=== CENSUS ===
[census_ddw_pca2905] 1068x94 -> 1068x94 (dropped 0 dup rows, 0 empty cols)
  -> saved census_ddw_pca2905: 1068 rows x 94 cols -> /content/drive/MyDrive/Colab Notebooks/Data_Organized/Cleaned/census_ddw_pca2905.csv
[census_pca_cdb_2929] 1066x96 -> 1066x96 (dropped 0 dup rows, 0 empty cols)
  -> saved census_pca_cdb_2929: 1066 rows x 96 cols -> /content/drive/MyDrive/Colab Notebooks/Data_Organized/Cleaned/census_pca_cdb_2929.csv
[raichur_village_to_gp_mapping] 867x6 -> 867x6 (dropped 0 dup rows, 0 empty cols)
  -> saved raichur_village_to_gp_mapping: 867 rows x 6 cols -> /content/drive/MyDrive/Colab Notebooks/Data_Organized/Cleaned/raichur_village_to_gp_mapping.csv
[raichur_gp_level_census2011] 179x88 -> 179x88 (dropped 0 dup rows, 0 empty cols)
  -> saved raichur_gp_level_census2011: 179 rows x 88 cols -> /content/drive/MyDrive/Colab Notebooks/Data_Organized/Cleaned/raichur_gp_level_census2011.csv
[raichur_villages_unmatched] 2x3 -> 2x3 (dropped 0 dup rows, 0 empty cols)

In [ ]:
# Mount Google Drive so BASE_DIR can point at your uploaded Data_Organized folder
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [ ]:
# ============================================================
# PHASE 1 — GP-LEVEL INDICATOR JOIN
# ============================================================

import pandas as pd
import numpy as np
import os

# Your actual Google Drive location
BASE_DIR = "/content/drive/MyDrive/Colab Notebooks/Data_Organized"

CLEAN_DIR = os.path.join(BASE_DIR, "Cleaned")

print("Cleaned folder:")
print(CLEAN_DIR)

if not os.path.exists(CLEAN_DIR):
    raise FileNotFoundError("Cleaned folder was not found.")

print("\n✅ Cleaned folder found!")

Cleaned folder:
/content/drive/MyDrive/Colab Notebooks/Data_Organized/Cleaned

✅ Cleaned folder found!


In [ ]:
# ============================================================
# LOAD CLEANED DATASETS
# ============================================================

census = pd.read_csv(
    os.path.join(CLEAN_DIR, "raichur_gp_level_census2011.csv")
)

education = pd.read_csv(
    os.path.join(CLEAN_DIR, "education.csv"),
    low_memory=False
)

sanitation = pd.read_csv(
    os.path.join(CLEAN_DIR, "sanitation_and_infrastructure.csv"),
)

finance_receipts = pd.read_csv(
    os.path.join(CLEAN_DIR, "devadurga_receipts_payments_panel.csv")
)

finance_xvfc = pd.read_csv(
    os.path.join(CLEAN_DIR, "devadurga_xvfc_finance_panel.csv")
)

gpdp = pd.read_csv(
    os.path.join(CLEAN_DIR, "gp_pdp_scores_summary.csv")
)

gpdp_master = pd.read_csv(
    os.path.join(CLEAN_DIR, "devadurga_gpdp_master.csv")
)

print("Census:", census.shape)
print("Education:", education.shape)
print("Sanitation:", sanitation.shape)
print("Finance receipts:", finance_receipts.shape)
print("Finance XVFC:", finance_xvfc.shape)
print("GPDP:", gpdp.shape)
print("GPDP master:", gpdp_master.shape)

Census: (179, 88)
Education: (588, 384)
Sanitation: (6008, 98)
Finance receipts: (132, 5)
Finance XVFC: (132, 9)
GPDP: (33, 6)
GPDP master: (33, 6)


In [ ]:
# ============================================================
# CHECK POSSIBLE GP IDENTIFIER COLUMNS
# ============================================================

datasets = {
    "Census": census,
    "Education": education,
    "Sanitation": sanitation,
    "Finance Receipts": finance_receipts,
    "Finance XVFC": finance_xvfc,
    "GPDP": gpdp,
    "GPDP Master": gpdp_master
}

for name, df in datasets.items():

    print("\n" + "=" * 60)
    print(name)
    print("=" * 60)

    possible_keys = [
        c for c in df.columns
        if any(word in c.lower()
               for word in [
                   "gp",
                   "gram panchayat",
                   "panchayat",
                   "lgd",
                   "code"
               ])
    ]

    print("Possible key columns:")
    for c in possible_keys:
        print("  -", c)


Census
Possible key columns:
  - gp_lgd_code
  - gp_name

Education
Possible key columns:
  - State Code
  - District Code
  - Sub District Code
  - Village Code
  - CD Block Code
  - Gram Panchayat Name
  - (If Pre-Primary School (Nursery/LKG/UKG not available within the village, the distance range code of nearest place where facility is available is given viz; a for < 5 Kms, b for 5-10 Kms and c for 10+ kms ).
  - (If Primary School not available within the village, the distance range code of nearest place where facility is available is given viz; a for < 5 Kms, b for 5-10 Kms and c for 10+ kms ).
  - (If Middle School not available within the village, the distance range code of nearest place where facility is available is given viz; a for < 5 Kms, b for 5-10 Kms and c for 10+ kms ).
  - (If Secondary School not available within the village, the distance range code of nearest place where facility is available is given viz; a for < 5 Kms, b for 5-10 Kms and c for 10+ kms ).
  - (If S

In [ ]:
# ============================================================
# STEP 4 — INSPECT ACTUAL GP IDENTIFIERS
# ============================================================

print("\n========== CENSUS ==========")
print(census[["gp_lgd_code", "gp_name"]].head(10).to_string(index=False))

print("\n========== SANITATION ==========")
print(sanitation[["gp_code", "gp_name"]].head(10).to_string(index=False))

print("\n========== FINANCE RECEIPTS ==========")
print(finance_receipts[["gp_lgd_code", "gp_name"]].head(10).to_string(index=False))

print("\n========== FINANCE XVFC ==========")
print(finance_xvfc[["gp_lgd_code", "gp_name"]].head(10).to_string(index=False))

print("\n========== GPDP MASTER ==========")
print(gpdp_master[["GP Code", "GP Name"]].head(10).to_string(index=False))

print("\n========== GPDP SUMMARY ==========")
print(gpdp[["Gram Panchayat and Equivalent"]].head(10).to_string(index=False))

print("\n========== EDUCATION ==========")
print(education[["Gram Panchayat Name"]].head(10).to_string(index=False))


========== CENSUS ==========
 gp_lgd_code             gp_name
      219671               Alkod
      219672              Arkera
      219673           B.Ganekal
      219674           Chinchodi
      219675           Dondambli
      219676              Gabbur
      219677              Galaga
      219678 Ganadhal (Deodurga)
      219679               Gugal
      219680             Hemanal

========== SANITATION ==========
 gp_code       gp_name
  215148        Adagal
  273085       Alur-Sk
  215149        Anawal
  215150         Belur
  215151   Cholachagud
  215152 Fakirbhudihal
  273086      Haligeri
  215154      Halkurki
  215156       Hebbali
  215157         Hosur

========== FINANCE RECEIPTS ==========
 gp_lgd_code             gp_name
      219671               Alkod
      273065            Amarapur
      219672              Arkera
      219673           B.Ganekal
      273066        Bhumanagunda
      219674           Chinchodi
      219675           Dondambli
      219676    

In [ ]:
# ============================================================
# STEP 5 — CHECK HOW MANY ROWS EACH GP HAS
# ============================================================

print("\n========== CENSUS ==========")
print("Total rows:", len(census))
print("Unique GPs:", census["gp_lgd_code"].nunique())

print("\n========== EDUCATION ==========")
print("Total rows:", len(education))
print("Unique GP names:", education["Gram Panchayat Name"].nunique())

print("\nRows per Education GP:")
print(
    education.groupby("Gram Panchayat Name")
    .size()
    .describe()
)

print("\nTop Education GPs by number of rows:")
print(
    education["Gram Panchayat Name"]
    .value_counts()
    .head(15)
)

print("\n========== SANITATION ==========")
print("Total rows:", len(sanitation))
print("Unique GP codes:", sanitation["gp_code"].nunique())

print("\nRows per Sanitation GP:")
print(
    sanitation.groupby("gp_code")
    .size()
    .describe()
)

print("\nTop Sanitation GPs by number of rows:")
print(
    sanitation["gp_code"]
    .value_counts()
    .head(15)
)


========== CENSUS ==========
Total rows: 179
Unique GPs: 179

========== EDUCATION ==========
Total rows: 588
Unique GP names: 122

Rows per Education GP:
count    122.000000
mean       4.754098
std        3.868310
min        1.000000
25%        2.000000
50%        4.000000
75%        7.000000
max       19.000000
dtype: float64

Top Education GPs by number of rows:
Gram Panchayat Name
HUSKUR                19
CHUNCHANAKUPPE        16
CHOLANAIKANAHALLI     15
MANDUR                14
MUGALUR               14
SHIVAKOTE             12
MANTAPA               12
HANDENAHALLI          12
BANDI KODIGE HALLI    12
CHIKKANA HALLI        11
VANAKANAHALLI         11
NERIGA                11
SURAGAJAKKANAHALLI    10
BYAGADADENAHALLI      10
ARAKERE               10
Name: count, dtype: int64

========== SANITATION ==========
Total rows: 6008
Unique GP codes: 6005

Rows per Sanitation GP:
count    6005.000000
mean        1.000500
std         0.022348
min         1.000000
25%         1.000000
50%    

In [ ]:
# ============================================================
# STEP 6 — INSPECT EDUCATION + SANITATION DATA
# ============================================================

print("\n========== EDUCATION COLUMNS ==========")

for i, col in enumerate(education.columns):
    print(i, ":", col)


print("\n========== EDUCATION SAMPLE ==========")
print(
    education[
        ["Gram Panchayat Name"]
    ].head(20).to_string(index=False)
)


print("\n========== SANITATION COLUMNS ==========")

for i, col in enumerate(sanitation.columns):
    print(i, ":", col)


print("\n========== SANITATION SAMPLE ==========")

print(
    sanitation[
        ["state_code", "district_code", "block_code",
         "gp_code", "gp_name", "csc_panchayat"]
    ].head(20).to_string(index=False)
)


print("\n========== SANITATION DUPLICATE GP CODES ==========")

duplicate_gp_codes = (
    sanitation["gp_code"]
    .value_counts()
    .loc[lambda x: x > 1]
)

print(duplicate_gp_codes)


========== EDUCATION COLUMNS ==========
0 : State Code
1 : State Name
2 : District Code
3 : District Name
4 : Sub District Code
5 : Sub District Name
6 : Village Code
7 : Village Name
8 : CD Block Code
9 : CD Block Name
10 : Gram Panchayat Name
11 : Reference Year
12 : Sub District Head Quarter (Name)
13 : Sub District Head Quarter (Distance in km)
14 : District Head Quarter (Name)
15 : District Head Quarter (Distance in km)
16 : Nearest Statutory Town (Name)
17 : Nearest Statutory Town (Distance in km)
18 : Within the State/UT (Name)
19 : Within the State/UT (Distance in km)
20 : Total Geographical Area (in Hectares)
21 : Total Households
22 : Total Population of Village
23 : Total Male Population of Village
24 : Total Female Population of Village
25 : Total Scheduled Castes Population of Village
26 : Total Scheduled Castes Male Population of Village
27 : Total Scheduled Castes Female Population of Village
28 : Total Scheduled Tribes Population of Village
29 : Total Scheduled Tribes 

In [ ]:
# ============================================================
# STEP 7 — CHECK GEOGRAPHICAL COVERAGE
# ============================================================

print("\n========== CENSUS ==========")

print("Districts:")
print(census["district_name"].value_counts(dropna=False).head(20)
      if "district_name" in census.columns else "No district_name column")

print("\nBlocks/Sub-districts:")
print(census["sub_district_name"].value_counts(dropna=False).head(20)
      if "sub_district_name" in census.columns else "No sub_district_name column")


print("\n========== EDUCATION ==========")

print("States:")
print(education["State Name"].value_counts(dropna=False).head(10))

print("\nDistricts:")
print(education["District Name"].value_counts(dropna=False).head(20))

print("\nSub-districts:")
print(education["Sub District Name"].value_counts(dropna=False).head(20))

print("\nCD Blocks:")
print(education["CD Block Name"].value_counts(dropna=False).head(20))


print("\n========== SANITATION ==========")

print("States:")
print(sanitation["state_name"].value_counts(dropna=False).head(10))

print("\nDistricts:")
print(sanitation["district_name"].value_counts(dropna=False).head(20))

print("\nBlocks:")
print(sanitation["block_name"].value_counts(dropna=False).head(20))


========== CENSUS ==========
Districts:
No district_name column

Blocks/Sub-districts:
No sub_district_name column

========== EDUCATION ==========
States:
State Name
KARNATAKA    588
Name: count, dtype: int64

Districts:
District Name
Bangalore    588
Name: count, dtype: int64

Sub-districts:
Sub District Name
Anekal             227
Bangalore North    199
Bangalore South    103
Bangalore East      59
Name: count, dtype: int64

CD Blocks:
CD Block Name
Anekal             227
Bangalore North    199
Bangalore South    103
Bangalore East      59
Name: count, dtype: int64

========== SANITATION ==========
States:
state_name
Karnataka    6008
Name: count, dtype: int64

Districts:
district_name
Belagavi          506
Tumakuru          330
Shivamogga        271
Hassan            267
Mysuru            266
Kalaburagi        263
Ballari           237
Mandya            233
Uttar Kannad      231
Dakshin Kannad    228
Chikkamagaluru    226
Haveri            223
Vijayapura        212
Bagalkot       

In [ ]:
# ============================================================
# STEP 7B — CHECK GP NAME OVERLAP
# ============================================================

census_names = set(
    census["gp_name"]
    .astype(str)
    .str.strip()
    .str.upper()
)

education_names = set(
    education["Gram Panchayat Name"]
    .astype(str)
    .str.strip()
    .str.upper()
)

sanitation_names = set(
    sanitation["gp_name"]
    .astype(str)
    .str.strip()
    .str.upper()
)

print("Census GPs:", len(census_names))
print("Education GPs:", len(education_names))
print("Sanitation GPs:", len(sanitation_names))

print("\n========== CENSUS ↔ EDUCATION ==========")
overlap_ce = census_names & education_names
print("Name overlap:", len(overlap_ce))
print(sorted(overlap_ce)[:50])

print("\n========== CENSUS ↔ SANITATION ==========")
overlap_cs = census_names & sanitation_names
print("Name overlap:", len(overlap_cs))
print(sorted(overlap_cs)[:50])

Census GPs: 177
Education GPs: 123
Sanitation GPs: 5482

========== CENSUS ↔ EDUCATION ==========
Name overlap: 0
[]

========== CENSUS ↔ SANITATION ==========
Name overlap: 177
['ALABANUR', 'ALKOD', 'AMADIHAL', 'AMARAPUR', 'AMEENAGADA', 'ANAHOSUR', 'ANKUSHADODDI', 'ANWARI', 'ARKERA', 'AROLI', 'ATHANUR', 'ATHKOOR', 'B.GANEKAL', 'BADARLI', 'BAGALAWAD', 'BAIDODDI', 'BALLATAGI', 'BANNIGOL', 'BAPPUR', 'BASSAPUR K.', 'BAYYAPUR', 'BHOGAVATHI', 'BHOOTALADINNI', 'BHUMANAGUNDA', 'BICHALI', 'BIJANGERA', 'BYAGAVATA', 'CHAGABHAVI', 'CHANDRABANDA', 'CHENNALLI', 'CHICKKOTNEKAL', 'CHICKSUGUR', 'CHINCHARAKI', 'CHINCHODI', 'CHITTAPURA', 'DEVARABHUPUR', 'DEVARAGUDI', 'DEVASUGUR', 'DHADESUGUR', 'DIGGANAYAKANABHAVI', 'DONDAMBLI', 'EACHANAL', 'GABBUR', 'GALAGA', 'GANADHAL', 'GANADHAL (DEODURGA)', 'GANADINNI', 'GANDHINAGAR', 'GEJJALAGATTA', 'GILLESUGUR']


In [ ]:
# ============================================================
# STEP 8 — CHECK CENSUS GP NAME DUPLICATES
# ============================================================

census_check = census.copy()

census_check["gp_name_clean"] = (
    census_check["gp_name"]
    .astype(str)
    .str.strip()
    .str.upper()
)

print("Total Census rows:", len(census_check))
print("Unique cleaned GP names:", census_check["gp_name_clean"].nunique())

print("\n========== DUPLICATE GP NAMES ==========")

duplicate_names = (
    census_check["gp_name_clean"]
    .value_counts()
    .loc[lambda x: x > 1]
)

print(duplicate_names)

print("\n========== DUPLICATE GP DETAILS ==========")

if len(duplicate_names) > 0:
    print(
        census_check[
            census_check["gp_name_clean"].isin(duplicate_names.index)
        ][["gp_lgd_code", "gp_name", "gp_name_clean"]]
        .sort_values("gp_name_clean")
        .to_string(index=False)
    )

Total Census rows: 179
Unique cleaned GP names: 177

========== DUPLICATE GP NAMES ==========
gp_name_clean
GOREBAL     2
GUNJALLI    2
Name: count, dtype: int64

========== DUPLICATE GP DETAILS ==========
 gp_lgd_code  gp_name gp_name_clean
      219708  Gorebal       GOREBAL
      219811  Gorebal       GOREBAL
      219779 Gunjalli      GUNJALLI
      219815 Gunjalli      GUNJALLI


In [ ]:
# ============================================================
# STEP 8B — CHECK RAICHUR SANITATION MATCHES
# ============================================================

census_names = set(census_check["gp_name_clean"])

sanitation_check = sanitation.copy()

sanitation_check["gp_name_clean"] = (
    sanitation_check["gp_name"]
    .astype(str)
    .str.strip()
    .str.upper()
)

raichur_sanitation = sanitation_check[
    sanitation_check["gp_name_clean"].isin(census_names)
].copy()

print("Census GPs:", len(census))
print("Matching sanitation rows:", len(raichur_sanitation))
print(
    "Unique matching sanitation GPs:",
    raichur_sanitation["gp_name_clean"].nunique()
)

print("\n========== SANITATION MATCH SAMPLE ==========")

print(
    raichur_sanitation[
        ["gp_code", "gp_name", "gp_name_clean",
         "district_name", "block_name"]
    ]
    .head(30)
    .to_string(index=False)
)

Census GPs: 179
Matching sanitation rows: 196
Unique matching sanitation GPs: 177

========== SANITATION MATCH SAMPLE ==========
 gp_code             gp_name       gp_name_clean  district_name     block_name
  215160             Jalihal             JALIHAL       Bagalkot         Badami
  215305             Nagaral             NAGARAL       Bagalkot         Mudhol
  216241           Ramadurga           RAMADURGA        Ballari        Kudligi
  215821             Nagaral             NAGARAL       Belagavi        Chikodi
  216434            Yadlapur            YADLAPUR          Bidar          Bidar
  216450              Kallur              KALLUR          Bidar       Humnabad
  216686            Honnalli            HONNALLI Chamarajanagar Chamarajanagar
  217627              Mustur              MUSTUR      Davangere        Jagalur
  217645         Karadigudda         KARADIGUDDA        Dharwad        Dharwad
  273001           Somalapur           SOMALAPUR         Haveri     Ranebennur
  

In [ ]:
# STEP 8C — FILTER SANITATION TO RAICHUR AND CHECK MATCHING

census_check = census.copy()
census_check["gp_name_clean"] = (
    census_check["gp_name"]
    .astype(str)
    .str.strip()
    .str.upper()
)

sanitation_check = sanitation.copy()
sanitation_check["gp_name_clean"] = (
    sanitation_check["gp_name"]
    .astype(str)
    .str.strip()
    .str.upper()
)

# Keep ONLY Raichur district
raichur_sanitation = sanitation_check[
    sanitation_check["district_name"]
    .astype(str)
    .str.strip()
    .str.upper()
    == "RAICHUR"
].copy()

print("========== RAICHUR SANITATION COVERAGE ==========")
print("Census rows:", len(census_check))
print("Raichur sanitation rows:", len(raichur_sanitation))
print("Unique sanitation GP codes:", raichur_sanitation["gp_code"].nunique())
print("Unique sanitation GP names:", raichur_sanitation["gp_name_clean"].nunique())

# Compare names
census_names = set(census_check["gp_name_clean"])
sanitation_names = set(raichur_sanitation["gp_name_clean"])

print("\n========== NAME MATCH ==========")
print("Census unique names:", len(census_names))
print("Sanitation unique names:", len(sanitation_names))
print("Census names missing in sanitation:")
print(sorted(census_names - sanitation_names))

print("\nSanitation names not in Census:")
print(sorted(sanitation_names - census_names))

# Duplicate sanitation GP codes within Raichur
print("\n========== DUPLICATE SANITATION GP CODES ==========")
duplicates = raichur_sanitation[
    raichur_sanitation.duplicated("gp_code", keep=False)
]

if len(duplicates) == 0:
    print("No duplicate GP codes within Raichur.")
else:
    print(
        duplicates[
            ["gp_code", "gp_name", "district_name", "block_name"]
        ].sort_values("gp_code").to_string(index=False)
    )

# Show Raichur sanitation records
print("\n========== RAICHUR SANITATION SAMPLE ==========")
print(
    raichur_sanitation[
        ["gp_code", "gp_name", "district_name", "block_name"]
    ].head(30).to_string(index=False)
)

========== RAICHUR SANITATION COVERAGE ==========
Census rows: 179
Raichur sanitation rows: 179
Unique sanitation GP codes: 179
Unique sanitation GP names: 177

========== NAME MATCH ==========
Census unique names: 177
Sanitation unique names: 177
Census names missing in sanitation:
[]

Sanitation names not in Census:
[]

========== DUPLICATE SANITATION GP CODES ==========
No duplicate GP codes within Raichur.

========== RAICHUR SANITATION SAMPLE ==========
 gp_code             gp_name district_name block_name
  219671               Alkod       Raichur  Devadurga
  273065            Amarapur       Raichur  Devadurga
  219672              Arkera       Raichur  Devadurga
  219673           B.Ganekal       Raichur  Devadurga
  273066        Bhumanagunda       Raichur  Devadurga
  219674           Chinchodi       Raichur  Devadurga
  219675           Dondambli       Raichur  Devadurga
  219676              Gabbur       Raichur  Devadurga
  219677              Galaga       Raichur  Devadur

In [ ]:
# STEP 9 — CHECK FINANCE + GPDP COVERAGE AGAINST CENSUS

census_codes = set(
    census["gp_lgd_code"]
    .dropna()
    .astype(int)
)

finance_codes = set(
    finance_receipts["gp_lgd_code"]
    .dropna()
    .astype(int)
)

gpdp_codes = set(
    gpdp_master["GP Code"]
    .dropna()
    .astype(int)
)

print("========== COVERAGE ==========")

print("Census GPs:", len(census_codes))
print("Finance GPs:", len(finance_codes))
print("GPDP GPs:", len(gpdp_codes))

print("\nFinance ∩ Census:", len(finance_codes & census_codes))
print("GPDP ∩ Census:", len(gpdp_codes & census_codes))

print("\nFinance ∩ GPDP:", len(finance_codes & gpdp_codes))

print("\n========== FINANCE GPs NOT IN CENSUS ==========")
print(sorted(finance_codes - census_codes))

print("\n========== GPDP GPs NOT IN CENSUS ==========")
print(sorted(gpdp_codes - census_codes))

print("\n========== CENSUS GPs WITHOUT FINANCE ==========")
print(len(census_codes - finance_codes))

print("\n========== CENSUS GPs WITHOUT GPDP ==========")
print(len(census_codes - gpdp_codes))

========== COVERAGE ==========
Census GPs: 179
Finance GPs: 33
GPDP GPs: 33

Finance ∩ Census: 33
GPDP ∩ Census: 33

Finance ∩ GPDP: 33

========== FINANCE GPs NOT IN CENSUS ==========
[]

========== GPDP GPs NOT IN CENSUS ==========
[]

========== CENSUS GPs WITHOUT FINANCE ==========
146

========== CENSUS GPs WITHOUT GPDP ==========
146


In [ ]:
# STEP 10 — CREATE THE COMMON 33-GP SET

common_codes = (
    census_codes
    & finance_codes
    & gpdp_codes
)

print("Common GPs across Census + Finance + GPDP:", len(common_codes))

devadurga_census = census[
    census["gp_lgd_code"].isin(common_codes)
].copy()

print("\nDevadurga Census rows:", len(devadurga_census))

print("\n========== DEVADURGA GP LIST ==========")

print(
    devadurga_census[
        ["gp_lgd_code", "gp_name"]
    ]
    .sort_values("gp_lgd_code")
    .to_string(index=False)
)

Common GPs across Census + Finance + GPDP: 33

Devadurga Census rows: 33

========== DEVADURGA GP LIST ==========
 gp_lgd_code             gp_name
      219671               Alkod
      219672              Arkera
      219673           B.Ganekal
      219674           Chinchodi
      219675           Dondambli
      219676              Gabbur
      219677              Galaga
      219678 Ganadhal (Deodurga)
      219679               Gugal
      219680             Hemanal
      219681           Hirebudur
      219682      Hosur Siddapur
      219683   Jagirajadaladinni
      219684           Jalahalli
      219685          Jeerabandi
      219686          K.Irabgera
      219687         Karadigudda
      219688           Karigudda
      219689              Koppar
      219690          Kothadoddi
      219691           Kyadigera
      219692           Maladakal
      219693            Masarkal
      219694           Mundaragi
      219695              Mustur
      219696         Nagadad

In [ ]:
# STEP 11 — INSPECT FINANCE AND GPDP VARIABLES

print("========== FINANCE RECEIPTS ==========")
print("Shape:", finance_receipts.shape)
print("\nColumns:")
print(finance_receipts.columns.tolist())

print("\nSample:")
print(finance_receipts.head(10).to_string(index=False))


print("\n\n========== FINANCE XVFC ==========")
print("Shape:", finance_xvfc.shape)
print("\nColumns:")
print(finance_xvfc.columns.tolist())

print("\nSample:")
print(finance_xvfc.head(10).to_string(index=False))


print("\n\n========== GPDP MASTER ==========")
print("Shape:", gpdp_master.shape)
print("\nColumns:")
print(gpdp_master.columns.tolist())

print("\nSample:")
print(gpdp_master.head(10).to_string(index=False))

========== FINANCE RECEIPTS ==========
Shape: (132, 5)

Columns:
['receipts', 'payments', 'financial_year', 'gp_name', 'gp_lgd_code']

Sample:
  receipts  payments financial_year             gp_name  gp_lgd_code
14530119.0 2871469.0        2022-23               Alkod       219671
 9524406.0 6029923.0        2022-23            Amarapur       273065
 9400658.0 4671443.0        2022-23              Arkera       219672
 9979317.0 4943120.0        2022-23           B.Ganekal       219673
11952732.0 4151811.0        2022-23        Bhumanagunda       273066
10983005.0 6717157.0        2022-23           Chinchodi       219674
13798223.0 7579128.0        2022-23           Dondambli       219675
17006362.0 5570561.0        2022-23              Gabbur       219676
12149788.0 5708276.0        2022-23              Galaga       219677
11532510.0 6790605.0        2022-23 Ganadhal (Deodurga)       219678


========== FINANCE XVFC ==========
Shape: (132, 9)

Columns:
['gp_name', 'RV', 'PV', 'CV', 'JV',

In [ ]:
# STEP 12 — CHECK FINANCIAL YEAR COVERAGE

print("========== FINANCE RECEIPTS YEARS ==========")
print(finance_receipts["financial_year"].value_counts().sort_index())

print("\n========== FINANCE XVFC YEARS ==========")
print(finance_xvfc["financial_year"].value_counts().sort_index())

print("\n========== GPDP SCORE DISTRIBUTION ==========")
print(gpdp_master["GPDP Score"].describe())

print("\n========== GPDP GRADES ==========")
print(gpdp_master["Grade"].value_counts())

print("\n========== GPDP SCORE + GRADE ==========")
print(
    gpdp_master[
        ["GP Name", "GP Code", "GPDP Score", "Grade"]
    ]
    .sort_values("GPDP Score")
    .to_string(index=False)
)

========== FINANCE RECEIPTS YEARS ==========
financial_year
2022-23    33
2023-24    33
2024-25    33
2025-26    33
Name: count, dtype: int64

========== FINANCE XVFC YEARS ==========
financial_year
2022-23    33
2023-24    33
2024-25    33
2025-26    33
Name: count, dtype: int64

========== GPDP SCORE DISTRIBUTION ==========
count    33.000000
mean     34.848485
std       1.523179
min      30.000000
25%      35.000000
50%      35.000000
75%      35.000000
max      40.000000
Name: GPDP Score, dtype: float64

========== GPDP GRADES ==========
Grade
D    33
Name: count, dtype: int64

========== GPDP SCORE + GRADE ==========
            GP Name  GP Code  GPDP Score Grade
              Alkod   219671          30     D
          Maladakal   219692          30     D
             Arkera   219672          35     D
           Amarapur   273065          35     D
       Bhumanagunda   273066          35     D
          Chinchodi   219674          35     D
          Dondambli   219675          35 

In [ ]:
# STEP 13 — INSPECT CENSUS COLUMNS

print("Census shape:", census.shape)

for i, col in enumerate(census.columns):
    print(i, ":", col)

Census shape: (179, 88)
0 : gp_lgd_code
1 : gp_name
2 : No_HH
3 : TOT_P
4 : TOT_M
5 : TOT_F
6 : P_06
7 : M_06
8 : F_06
9 : P_SC
10 : M_SC
11 : F_SC
12 : P_ST
13 : M_ST
14 : F_ST
15 : P_LIT
16 : M_LIT
17 : F_LIT
18 : P_ILL
19 : M_ILL
20 : F_ILL
21 : TOT_WORK_P
22 : TOT_WORK_M
23 : TOT_WORK_F
24 : MAINWORK_P
25 : MAINWORK_M
26 : MAINWORK_F
27 : MAIN_CL_P
28 : MAIN_CL_M
29 : MAIN_CL_F
30 : MAIN_AL_P
31 : MAIN_AL_M
32 : MAIN_AL_F
33 : MAIN_HH_P
34 : MAIN_HH_M
35 : MAIN_HH_F
36 : MAIN_OT_P
37 : MAIN_OT_M
38 : MAIN_OT_F
39 : MARGWORK_P
40 : MARGWORK_M
41 : MARGWORK_F
42 : MARG_CL_P
43 : MARG_CL_M
44 : MARG_CL_F
45 : MARG_AL_P
46 : MARG_AL_M
47 : MARG_AL_F
48 : MARG_HH_P
49 : MARG_HH_M
50 : MARG_HH_F
51 : MARG_OT_P
52 : MARG_OT_M
53 : MARG_OT_F
54 : MARGWORK_3_6_P
55 : MARGWORK_3_6_M
56 : MARGWORK_3_6_F
57 : MARG_CL_3_6_P
58 : MARG_CL_3_6_M
59 : MARG_CL_3_6_F
60 : MARG_AL_3_6_P
61 : MARG_AL_3_6_M
62 : MARG_AL_3_6_F
63 : MARG_HH_3_6_P
64 : MARG_HH_3_6_M
65 : MARG_HH_3_6_F
66 : MARG_OT_3_6_P
67

In [ ]:
# STEP 14 — CHECK CENSUS CANDIDATE INDICATORS

census_candidates = devadurga_census[
    [
        "gp_lgd_code",
        "gp_name",
        "No_HH",
        "TOT_P",
        "P_06",
        "P_SC",
        "P_ST",
        "P_LIT",
        "P_ILL",
        "TOT_WORK_P",
        "MARGWORK_P",
        "NON_WORK_P"
    ]
].copy()

print("========== CENSUS CANDIDATE INDICATORS ==========")
print(census_candidates.describe().T)

print("\n========== MISSING VALUES ==========")
print(census_candidates.isna().sum())

print("\n========== SAMPLE ==========")
print(census_candidates.head(10).to_string(index=False))

========== CENSUS CANDIDATE INDICATORS ==========
             count           mean           std       min       25%       50%  \
gp_lgd_code   33.0  227772.757576  19437.079029  219671.0  219679.0  219687.0   
No_HH         33.0    1305.000000    340.798327     769.0     998.0    1289.0   
TOT_P         33.0    7583.060606   2132.740885    4604.0    5835.0    7524.0   
P_06          33.0    1251.090909    382.113315     671.0     936.0    1196.0   
P_SC          33.0    1671.030303    973.390931     323.0    1200.0    1548.0   
P_ST          33.0    2640.636364   1435.290254     151.0    1487.0    2604.0   
P_LIT         33.0    2992.030303    975.648678    1657.0    2306.0    2814.0   
P_ILL         33.0    4591.030303   1457.203775    2690.0    3493.0    4425.0   
TOT_WORK_P    33.0    3870.303030    983.851726    2563.0    3260.0    3582.0   
MARGWORK_P    33.0     717.757576    386.436284     234.0     452.0     617.0   
NON_WORK_P    33.0    3712.757576   1242.819904    1933.0  

In [ ]:
# STEP 15 — INSPECT SANITATION INDICATORS FOR DEVADURGA

devadurga_sanitation = raichur_sanitation[
    raichur_sanitation["gp_code"].isin(common_codes)
].copy()

print("Devadurga sanitation rows:", len(devadurga_sanitation))

print("\n========== SANITATION COLUMNS ==========")
for i, col in enumerate(devadurga_sanitation.columns):
    print(i, ":", col)

print("\n========== MISSING VALUES ==========")
print(
    devadurga_sanitation.isna().sum()
    .sort_values(ascending=False)
    .head(30)
)

print("\n========== SAMPLE ==========")
print(
    devadurga_sanitation.head(5).to_string(index=False)
)

Devadurga sanitation rows: 33

========== SANITATION COLUMNS ==========
0 : _id
1 : id
2 : year
3 : state_name
4 : state_code
5 : district_name
6 : district_code
7 : block_name
8 : block_code
9 : gp_name
10 : gp_code
11 : villages_surveyed
12 : tot_pop
13 : pop_male
14 : pop_female
15 : tot_hh
16 : bank
17 : bank_dist_l1km
18 : bank_dist_2km
19 : bank_dist_5km
20 : bank_dist_10km
21 : bank_dist_g10_km
22 : bc_w_internet
23 : atm
24 : internet_bb
25 : all_weather_road
26 : cc_road_full
27 : cc_road_partial
28 : cc_road_not
29 : pub_trans_bus
30 : pub_trans_van
31 : pub_trans_auto
32 : pub_trans_no
33 : railway
34 : csc_panchayat
35 : csc_seperate
36 : csc_not
37 : elec_domes_1hrs
38 : elec_domes_4hrs
39 : elec_domes_8hrs
40 : elec_domes_12hrs
41 : elec_domes_no
42 : elec_msme
43 : pds
44 : mandi
45 : reg_mkt
46 : weekly_haat
47 : mkt_na
48 : tap_100
49 : tap_50
50 : tap_l50
51 : tap_one_hab
52 : tap_na
53 : tel_ll
54 : tel_mob
55 : tel_both
56 : tel_na
57 : clean_energy_hhs
58 : solar_w

In [ ]:
# STEP 16 — INSPECT CANDIDATE SANITATION INDICATORS

candidate_cols = [
    "villages_surveyed",
    "tot_pop",
    "tot_hh",

    # Roads / transport
    "all_weather_road",
    "cc_road_full",
    "cc_road_partial",
    "cc_road_not",
    "pub_trans_no",

    # Electricity
    "elec_domes_12hrs",
    "elec_domes_no",

    # Drinking water
    "tap_100",
    "tap_50",
    "tap_l50",
    "tap_one_hab",
    "tap_na",

    # Communication
    "tel_na",

    # Housing / energy
    "clean_energy_hhs",
    "kuccha_hhs",

    # Education facilities
    "prim_school",
    "middle_school",
    "high_school",
    "high_second_school",
    "school_college_na",

    # Health facilities
    "chc",
    "phc",
    "subcentre",
    "no_health",

    # Drainage
    "closed_drain",
    "pucca_slab_drain",
    "pucca_uncovered_drain",
    "kuchha_drain",
    "no_drain"
]

print("========== CANDIDATE INDICATORS ==========")

for col in candidate_cols:
    if col in devadurga_sanitation.columns:
        print(f"\n{col}")
        print("  dtype:", devadurga_sanitation[col].dtype)
        print("  min:", devadurga_sanitation[col].min())
        print("  max:", devadurga_sanitation[col].max())
        print("  mean:", round(devadurga_sanitation[col].mean(), 2))
        print("  unique:", devadurga_sanitation[col].nunique())

print("\n========== COMPLETE MISSING CHECK ==========")
print(
    devadurga_sanitation[candidate_cols]
    .isna()
    .sum()
    .sort_values(ascending=False)
)

print("\n========== CANDIDATE DATA ==========")
print(
    devadurga_sanitation[
        ["gp_code", "gp_name"] + candidate_cols
    ].to_string(index=False)
)

========== CANDIDATE INDICATORS ==========

villages_surveyed
  dtype: int64
  min: 1
  max: 10
  mean: 5.67
  unique: 10

tot_pop
  dtype: int64
  min: 4460
  max: 22735
  mean: 9198.27
  unique: 33

tot_hh
  dtype: int64
  min: 888
  max: 8908
  mean: 2173.82
  unique: 33

all_weather_road
  dtype: int64
  min: 0
  max: 8
  mean: 3.91
  unique: 8

cc_road_full
  dtype: int64
  min: 0
  max: 8
  mean: 0.52
  unique: 4

cc_road_partial
  dtype: int64
  min: 0
  max: 10
  mean: 2.85
  unique: 10

cc_road_not
  dtype: int64
  min: 0
  max: 8
  mean: 2.3
  unique: 9

pub_trans_no
  dtype: int64
  min: 0
  max: 6
  mean: 0.88
  unique: 6

elec_domes_12hrs
  dtype: int64
  min: 0
  max: 10
  mean: 1.91
  unique: 9

elec_domes_no
  dtype: int64
  min: 0
  max: 4
  mean: 0.3
  unique: 4

tap_100
  dtype: int64
  min: 0
  max: 7
  mean: 0.79
  unique: 5

tap_50
  dtype: int64
  min: 0
  max: 9
  mean: 3.18
  unique: 10

tap_l50
  dtype: int64
  min: 0
  max: 6
  mean: 0.88
  unique: 6

tap_one

In [ ]:
# STEP 17 — CREATE SELECTED SANITATION NEED INDICATORS

san = devadurga_sanitation.copy()

san["road_need"] = (
    san["cc_road_not"] / san["villages_surveyed"]
)

san["transport_need"] = (
    san["pub_trans_no"] / san["villages_surveyed"]
)

san["electricity_need"] = (
    san["elec_domes_no"] / san["villages_surveyed"]
)

san["water_need"] = (
    san["tap_na"] / san["villages_surveyed"]
)

san["kutcha_housing_need"] = (
    san["kuccha_hhs"] / san["tot_hh"]
)

san["health_access_need"] = (
    san["no_health"] / san["villages_surveyed"]
)

san["drainage_need"] = (
    san["no_drain"] / san["villages_surveyed"]
)

sanitation_indicators = san[
    [
        "gp_code",
        "gp_name",
        "road_need",
        "transport_need",
        "electricity_need",
        "water_need",
        "kutcha_housing_need",
        "health_access_need",
        "drainage_need"
    ]
].copy()

print("========== SANITATION NEED INDICATORS ==========")

print(
    sanitation_indicators
    .describe()
    .T
    .to_string()
)

print("\n========== COMPLETE TABLE ==========")

print(
    sanitation_indicators.to_string(index=False)
)

========== SANITATION NEED INDICATORS ==========
                     count           mean           std            min            25%           50%            75%           max
gp_code               33.0  227772.757576  19437.079029  219671.000000  219679.000000  219687.00000  219695.000000  273069.00000
road_need             33.0       0.401635      0.432457       0.000000       0.000000       0.20000       0.888889       1.00000
transport_need        33.0       0.140356      0.253312       0.000000       0.000000       0.00000       0.166667       1.00000
electricity_need      33.0       0.054834      0.183623       0.000000       0.000000       0.00000       0.000000       1.00000
water_need            33.0       0.137326      0.279477       0.000000       0.000000       0.00000       0.142857       1.00000
kutcha_housing_need   33.0       0.257240      0.172656       0.000179       0.126402       0.26314       0.363039       0.65554
health_access_need    33.0       0.628283      0

In [ ]:
# =========================================================
# STEP 18 — BUILD THE 33-GP MASTER INDICATOR TABLE
# =========================================================

# ---------------------------------------------------------
# 0. GET CENSUS DATAFRAME
# ---------------------------------------------------------

# clean_census() stores the GP-level Census dataframe as "gp_census"
census_gp = census_results["gp_census"].copy()


# ---------------------------------------------------------
# 1. CENSUS: Devadurga GPs
# ---------------------------------------------------------

census_dev = census_gp[
    census_gp["gp_lgd_code"].isin(common_codes)
].copy()

# Create Census-derived need indicators
census_dev["illiteracy_rate"] = census_dev["P_ILL"] / census_dev["TOT_P"]
census_dev["child_share"] = census_dev["P_06"] / census_dev["TOT_P"]
census_dev["marginal_worker_share"] = census_dev["MARGWORK_P"] / census_dev["TOT_P"]
census_dev["non_worker_share"] = census_dev["NON_WORK_P"] / census_dev["TOT_P"]

census_indicators = census_dev[
    ["gp_lgd_code", "gp_name", "illiteracy_rate", "child_share",
     "marginal_worker_share", "non_worker_share"]
].copy()

census_indicators = census_indicators.rename(columns={"gp_lgd_code": "gp_code"})


# ---------------------------------------------------------
# 2. SANITATION
# ---------------------------------------------------------

sanitation_final = sanitation_indicators.copy()


# ---------------------------------------------------------
# 3. FINANCE
# ---------------------------------------------------------

finance = finance_results["devadurga_receipts_payments_panel"].copy()
finance["receipts"] = pd.to_numeric(finance["receipts"], errors="coerce")
finance["payments"] = pd.to_numeric(finance["payments"], errors="coerce")

finance_avg = (
    finance
    .groupby("gp_lgd_code")
    .agg(avg_receipts=("receipts", "mean"), avg_payments=("payments", "mean"))
    .reset_index()
)
finance_avg = finance_avg.rename(columns={"gp_lgd_code": "gp_code"})


# ---------------------------------------------------------
# 4. ADD HOUSEHOLD COUNT FROM CENSUS
# ---------------------------------------------------------

households = census_dev[["gp_lgd_code", "No_HH"]].rename(
    columns={"gp_lgd_code": "gp_code", "No_HH": "tot_hh_census"}
)
finance_avg = finance_avg.merge(households, on="gp_code", how="left")


# ---------------------------------------------------------
# 5. FINANCE PER HOUSEHOLD
# ---------------------------------------------------------

finance_avg["receipts_per_hh"] = finance_avg["avg_receipts"] / finance_avg["tot_hh_census"]
finance_avg["payments_per_hh"] = finance_avg["avg_payments"] / finance_avg["tot_hh_census"]


# ---------------------------------------------------------
# 6. GPDP
# ---------------------------------------------------------

gpdp = gp_pdp_results["master"].copy()
gpdp_final = gpdp[["GP Code", "GP Name", "GPDP Score"]].rename(
    columns={"GP Code": "gp_code", "GP Name": "gp_name", "GPDP Score": "gpdp_score"}
)


# ---------------------------------------------------------
# 7. MERGE CENSUS + SANITATION
# ---------------------------------------------------------

master = census_indicators.merge(sanitation_final, on=["gp_code", "gp_name"], how="inner")


# ---------------------------------------------------------
# 8. MERGE FINANCE
# ---------------------------------------------------------

master = master.merge(
    finance_avg[["gp_code", "avg_receipts", "avg_payments", "receipts_per_hh", "payments_per_hh"]],
    on="gp_code", how="inner"
)


# ---------------------------------------------------------
# 9. MERGE GPDP
# ---------------------------------------------------------

master = master.merge(gpdp_final[["gp_code", "gpdp_score"]], on="gp_code", how="inner")


# ---------------------------------------------------------
# 10. CHECK RESULT
# ---------------------------------------------------------

print("========== MASTER TABLE ==========")
print("Rows:", len(master))
print("Columns:", len(master.columns))
print("\nColumns:")
for i, col in enumerate(master.columns):
    print(i, ":", col)
print("\nMissing values:")
print(master.isna().sum().sort_values(ascending=False))
print("\nDuplicate GP codes:")
print(master["gp_code"].duplicated().sum())
print("\n========== MASTER DATA ==========")
print(master.to_string(index=False))

========== MASTER TABLE ==========
Rows: 33
Columns: 18

Columns:
0 : gp_code
1 : gp_name
2 : illiteracy_rate
3 : child_share
4 : marginal_worker_share
5 : non_worker_share
6 : road_need
7 : transport_need
8 : electricity_need
9 : water_need
10 : kutcha_housing_need
11 : health_access_need
12 : drainage_need
13 : avg_receipts
14 : avg_payments
15 : receipts_per_hh
16 : payments_per_hh
17 : gpdp_score

Missing values:
gp_code                  0
gp_name                  0
illiteracy_rate          0
child_share              0
marginal_worker_share    0
non_worker_share         0
road_need                0
transport_need           0
electricity_need         0
water_need               0
kutcha_housing_need      0
health_access_need       0
drainage_need            0
avg_receipts             0
avg_payments             0
receipts_per_hh          0
payments_per_hh          0
gpdp_score               0
dtype: int64

Duplicate GP codes:
0

========== MASTER DATA ==========
 gp_code             g